In [1]:
"""
MNIST 숫자 분류 신경망 구현
===========================

학습 목표:
1. MNIST 데이터셋 로드 및 전처리
2. 신경망 모델 정의 (nn.Module)
3. 학습 루프 구현
4. 모델 평가 및 시각화
5. 다양한 아키텍처 비교

실행 전 설치:
pip install torch torchvision matplotlib numpy
"""

# ============================================
# 1. 라이브러리 import
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from time import time

print("=" * 60)
print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")
print("=" * 60)


# ============================================
# 2. 데이터 로드 및 전처리
# ============================================

print("\n" + "=" * 60)
print("섹션 1: MNIST 데이터셋 준비")
print("=" * 60)

# 2.1 데이터 변환 정의
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # PIL Image를 Tensor로
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 평균, 표준편차
    ]
)

# 2.2 데이터셋 다운로드
print("\n📌 데이터셋 다운로드 중...")

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"✅ 학습 데이터: {len(train_dataset)}개")
print(f"✅ 테스트 데이터: {len(test_dataset)}개")

# 2.3 데이터 로더 생성
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n배치 크기: {batch_size}")
print(f"학습 배치 수: {len(train_loader)}")
print(f"테스트 배치 수: {len(test_loader)}")

# 2.4 데이터 확인
print("\n📌 샘플 데이터 확인")

# 첫 번째 배치 가져오기
images, labels = next(iter(train_loader))
print(f"이미지 배치 shape: {images.shape}")  # [batch_size, 1, 28, 28]
print(f"레이블 배치 shape: {labels.shape}")  # [batch_size]

# 2.5 샘플 이미지 시각화
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    img = images[i].squeeze()  # [1, 28, 28] -> [28, 28]
    label = labels[i].item()

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Label: {label}")
    axes[i].axis("off")

plt.suptitle("MNIST Sample Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("mnist_samples.png", dpi=150, bbox_inches="tight")
plt.show()


# ============================================
# 3. 모델 정의
# ============================================

print("\n" + "=" * 60)
print("섹션 2: 신경망 모델 정의")
print("=" * 60)


# 3.1 간단한 Fully Connected Network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        # 28x28 = 784 입력
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)  # 10개 클래스 (0-9)

    def forward(self, x):
        # Flatten: [batch, 1, 28, 28] -> [batch, 784]
        x = x.view(-1, 784)

        # 은닉층 1
        x = F.relu(self.fc1(x))

        # 은닉층 2
        x = F.relu(self.fc2(x))

        # 출력층 (softmax는 loss에서 처리)
        x = self.fc3(x)
        return x


# 3.2 드롭아웃이 있는 모델
class SimpleNNWithDropout(nn.Module):
    def __init__(self):
        super(SimpleNNWithDropout, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


# 3.3 CNN 모델
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 합성곱 층
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14
        self.pool = nn.MaxPool2d(2, 2)  # 크기 1/2

        # Fully connected 층
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        # Conv1 + ReLU + Pool
        x = self.pool(F.relu(self.conv1(x)))  # [batch, 32, 14, 14]

        # Conv2 + ReLU + Pool
        x = self.pool(F.relu(self.conv2(x)))  # [batch, 64, 7, 7]

        # Flatten
        x = x.view(-1, 64 * 7 * 7)

        # FC1 + ReLU + Dropout
        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        # FC2 (출력)
        x = self.fc2(x)
        return x


# 모델 생성
print("\n📌 모델 아키텍처")

models_dict = {
    "Simple NN": SimpleNN(),
    "NN with Dropout": SimpleNNWithDropout(),
    "Simple CNN": SimpleCNN(),
}

for name, model in models_dict.items():
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{name}:")
    print(f"  전체 파라미터: {total_params:,}")
    print(f"  학습 가능 파라미터: {trainable_params:,}")

# 모델 구조 출력
print("\n📌 Simple CNN 상세 구조:")
print(models_dict["Simple CNN"])


# ============================================
# 4. 학습 함수 정의
# ============================================

print("\n" + "=" * 60)
print("섹션 3: 학습 및 평가 함수")
print("=" * 60)


def train_epoch(model, device, train_loader, optimizer, criterion, epoch):
    """한 에포크 학습"""
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # 데이터를 디바이스로 이동
        data, target = data.to(device), target.to(device)

        # Gradient 초기화
        optimizer.zero_grad()

        # Forward
        output = model(data)
        loss = criterion(output, target)

        # Backward
        loss.backward()

        # Update
        optimizer.step()

        # 통계
        running_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        total += target.size(0)

        # 진행상황 출력
        if (batch_idx + 1) % 100 == 0:
            print(
                f"  Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Acc: {100.0 * correct / total:.2f}%"
            )

    avg_loss = running_loss / len(train_loader)
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy


def test(model, device, test_loader, criterion):
    """테스트 평가"""
    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            # Loss 누적
            test_loss += criterion(output, target).item()

            # 정확도 계산
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)

    avg_loss = test_loss / len(test_loader)
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy


# ============================================
# 5. 모델 학습
# ============================================

print("\n" + "=" * 60)
print("섹션 4: 모델 학습")
print("=" * 60)

# 학습할 모델 선택
model_name = "Simple CNN"
model = models_dict[model_name].to(device)

print(f"\n🚀 {model_name} 학습 시작\n")

# 하이퍼파라미터
learning_rate = 0.001
epochs = 10

# Loss 함수와 Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 학습 기록
train_losses = []
train_accs = []
test_losses = []
test_accs = []

# 학습 루프
start_time = time()

for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")
    print("-" * 60)

    # 학습
    train_loss, train_acc = train_epoch(
        model, device, train_loader, optimizer, criterion, epoch
    )

    # 평가
    test_loss, test_acc = test(model, device, test_loader, criterion)

    # 기록
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    # 결과 출력
    print(f"\n📊 Epoch {epoch} 결과:")
    print(f"  학습 Loss: {train_loss:.4f} | 학습 정확도: {train_acc:.2f}%")
    print(f"  테스트 Loss: {test_loss:.4f} | 테스트 정확도: {test_acc:.2f}%")
    print()

end_time = time()
training_time = end_time - start_time

print("=" * 60)
print("✅ 학습 완료!")
print(f"⏱️  총 학습 시간: {training_time:.2f}초")
print(f"🎯 최종 테스트 정확도: {test_accs[-1]:.2f}%")
print("=" * 60)


# ============================================
# 6. 학습 결과 시각화
# ============================================

print("\n" + "=" * 60)
print("섹션 5: 결과 시각화")
print("=" * 60)

# 6.1 Loss와 Accuracy 그래프
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 그래프
ax = axes[0]
ax.plot(
    range(1, epochs + 1),
    train_losses,
    "b-",
    marker="o",
    label="Train Loss",
    linewidth=2,
)
ax.plot(
    range(1, epochs + 1), test_losses, "r-", marker="s", label="Test Loss", linewidth=2
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"{model_name} - Loss over Epochs")
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy 그래프
ax = axes[1]
ax.plot(
    range(1, epochs + 1),
    train_accs,
    "b-",
    marker="o",
    label="Train Accuracy",
    linewidth=2,
)
ax.plot(
    range(1, epochs + 1),
    test_accs,
    "r-",
    marker="s",
    label="Test Accuracy",
    linewidth=2,
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"{model_name} - Accuracy over Epochs")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()

# 6.2 예측 결과 시각화
print("\n📌 예측 결과 확인")

model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    predictions = outputs.argmax(dim=1)

# CPU로 이동
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# 샘플 20개 시각화
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.ravel()

for i in range(20):
    img = images[i].squeeze()
    true_label = labels[i].item()
    pred_label = predictions[i].item()

    axes[i].imshow(img, cmap="gray")

    # 정답이면 파란색, 오답이면 빨간색
    color = "blue" if true_label == pred_label else "red"
    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label}", color=color, fontweight="bold"
    )
    axes[i].axis("off")

plt.suptitle(f"{model_name} - Predictions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("predictions.png", dpi=150, bbox_inches="tight")
plt.show()


# ============================================
# 7. 혼동 행렬 (Confusion Matrix)
# ============================================

print("\n📌 혼동 행렬 생성")

from sklearn.metrics import confusion_matrix
import seaborn as sns

# 전체 테스트 데이터에 대한 예측
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 혼동 행렬 계산
cm = confusion_matrix(all_labels, all_preds)

# 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(10), yticklabels=range(10)
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"{model_name} - Confusion Matrix")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# 클래스별 정확도
print("\n📊 클래스별 정확도:")
class_correct = cm.diagonal()
class_total = cm.sum(axis=1)

for digit in range(10):
    accuracy = 100 * class_correct[digit] / class_total[digit]
    print(
        f"  숫자 {digit}: {accuracy:.2f}% ({class_correct[digit]}/{class_total[digit]})"
    )


# ============================================
# 8. 오분류 사례 분석
# ============================================

print("\n" + "=" * 60)
print("섹션 6: 오분류 사례 분석")
print("=" * 60)

# 오분류된 샘플 찾기
misclassified_indices = []
misclassified_images = []
misclassified_true = []
misclassified_pred = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        # 오분류 찾기
        incorrect = preds.ne(labels)
        incorrect_indices = incorrect.nonzero(as_tuple=True)[0]

        for idx in incorrect_indices:
            if len(misclassified_indices) < 20:  # 최대 20개
                misclassified_images.append(images[idx].cpu())
                misclassified_true.append(labels[idx].cpu().item())
                misclassified_pred.append(preds[idx].cpu().item())

print(f"총 오분류 샘플 수: {len(all_labels) - sum(class_correct)}")
print(f"표시할 샘플 수: {len(misclassified_images)}")

# 시각화
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.ravel()

for i in range(min(20, len(misclassified_images))):
    img = misclassified_images[i].squeeze()
    true_label = misclassified_true[i]
    pred_label = misclassified_pred[i]

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(
        f"True: {true_label} | Pred: {pred_label}", color="red", fontweight="bold"
    )
    axes[i].axis("off")

plt.suptitle("Misclassified Examples", fontsize=14, fontweight="bold", color="red")
plt.tight_layout()
plt.savefig("misclassified_examples.png", dpi=150, bbox_inches="tight")
plt.show()


# ============================================
# 9. 모델 저장 및 로드
# ============================================

print("\n" + "=" * 60)
print("섹션 7: 모델 저장 및 로드")
print("=" * 60)

# 모델 저장
model_path = "mnist_model.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epochs,
        "loss": test_losses[-1],
        "accuracy": test_accs[-1],
    },
    model_path,
)

print(f"✅ 모델 저장: {model_path}")

# 모델 로드
loaded_model = SimpleCNN().to(device)
checkpoint = torch.load(model_path)
loaded_model.load_state_dict(checkpoint["model_state_dict"])

print("✅ 모델 로드 완료")
print(f"   Epoch: {checkpoint['epoch']}")
print(f"   Loss: {checkpoint['loss']:.4f}")
print(f"   Accuracy: {checkpoint['accuracy']:.2f}%")

# 로드된 모델 테스트
test_loss, test_acc = test(loaded_model, device, test_loader, criterion)
print("\n🔍 로드된 모델 검증:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.2f}%")


# ============================================
# 10. 다양한 모델 비교
# ============================================

print("\n" + "=" * 60)
print("섹션 8: 모델 아키텍처 비교")
print("=" * 60)

# 각 모델을 5 에포크씩 학습하여 비교
comparison_results = {}

for model_name, model in models_dict.items():
    print(f"\n🔄 {model_name} 학습 중...")

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # 3 에포크만 빠르게 학습
    for epoch in range(1, 4):
        train_loss, train_acc = train_epoch(
            model, device, train_loader, optimizer, criterion, epoch
        )

    # 최종 평가
    test_loss, test_acc = test(model, device, test_loader, criterion)

    comparison_results[model_name] = {
        "test_loss": test_loss,
        "test_acc": test_acc,
        "params": sum(p.numel() for p in model.parameters()),
    }

    print(f"✅ {model_name}: {test_acc:.2f}%")

# 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(comparison_results.keys())
accuracies = [comparison_results[m]["test_acc"] for m in model_names]
params = [comparison_results[m]["params"] / 1000 for m in model_names]  # K 단위

# 정확도 비교
ax = axes[0]
colors = ["skyblue", "lightgreen", "salmon"]
bars = ax.bar(
    model_names, accuracies, color=colors, alpha=0.7, edgecolor="black", linewidth=2
)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Model Comparison - Accuracy")
ax.set_ylim([90, 100])
ax.grid(True, alpha=0.3, axis="y")

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{acc:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# 파라미터 수 비교
ax = axes[1]
bars = ax.bar(
    model_names, params, color=colors, alpha=0.7, edgecolor="black", linewidth=2
)
ax.set_ylabel("Parameters (K)")
ax.set_title("Model Comparison - Parameters")
ax.grid(True, alpha=0.3, axis="y")

for bar, param in zip(bars, params):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{param:.1f}K",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


# ============================================
# 11. 종합 정리
# ============================================

print("\n" + "=" * 60)
print("📚 핵심 개념 정리")
print("=" * 60)

print("""
1️⃣ 데이터 준비
   - transforms: 이미지 전처리 (정규화)
   - Dataset: 데이터 저장
   - DataLoader: 미니배치 생성

2️⃣ 모델 정의 (nn.Module)
   - __init__: 층(layer) 정의
   - forward: 순전파 정의
   - nn.Linear: Fully Connected Layer
   - nn.Conv2d: 합성곱 층
   - nn.MaxPool2d: 풀링 층

3️⃣ 학습 과정
   1) Forward: model(data)
   2) Loss: criterion(output, target)
   3) Backward: loss.backward()
   4) Update: optimizer.step()
   5) Zero grad: optimizer.zero_grad()

4️⃣ 평가
   - model.eval(): 평가 모드
   - torch.no_grad(): 기울기 계산 안 함
   - accuracy, confusion matrix

5️⃣ 모델 저장/로드
   - torch.save(): 체크포인트 저장
   - torch.load(): 모델 불러오기
""")

print("\n" + "=" * 60)
print("💡 실습 과제")
print("=" * 60)

print("""
1. Dropout 비율을 0.1, 0.3, 0.5로 바꿔가며 과적합 방지 효과 확인
2. 학습률을 0.0001, 0.001, 0.01로 조정하며 수렴 속도 비교
3. 배치 크기를 32, 64, 128로 바꿔가며 학습 속도 비교
4. 더 깊은 CNN 만들기 (Conv 층 3개 이상)
5. Data Augmentation 적용하기 (RandomRotation, RandomCrop)
6. Learning Rate Scheduler 사용하기
7. Early Stopping 구현하기
""")

print("\n✅ MNIST 분류 실습 완료!")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/megan/work/KCAI/kcai-pyspark/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/megan/work/KCAI/kcai-pyspark/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/megan/work/KCAI/kcai-pyspark/.venv/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 7

ModuleNotFoundError: No module named 'torchvision'